In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "225_build_silver_closed_bids_estimate_item_current.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.closed_bids_base_clean"
TARGET_TABLE = f"{catalog}.silver.closed_bids_estimate_item_current"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")

def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build table
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH estimate_rows AS (
        SELECT
              *
            , CASE
                  WHEN specification_description IS NOT NULL THEN specification_description
                  WHEN COALESCE(bid_code, '') LIKE '9602-%' THEN bid_item_description
                  WHEN COALESCE(bid_code, '') LIKE '9606-%' THEN bid_item_description
                  WHEN UPPER(COALESCE(bid_item_description, '')) LIKE '%PAYMENT ADJUSTMENT%' THEN bid_item_description
                  WHEN UPPER(COALESCE(bid_item_description, '')) LIKE '%SPECIAL DEDUCTION%' THEN bid_item_description
                  ELSE NULL
              END AS effective_specification_description
        FROM {SOURCE_TABLE}
        WHERE row_type = 'ENGINEER_ESTIMATE'
          AND is_engineer_estimate_row = TRUE
          AND project_id IS NOT NULL
    )

    , estimate_rows_keyed AS (
        SELECT
              *
            , CASE
                  WHEN specification_code IS NOT NULL
                   AND effective_specification_description IS NOT NULL
                   AND measurement_unit IS NOT NULL
                  THEN md5(
                      concat_ws(
                            '|'
                          , COALESCE(CAST(specification_code AS STRING), '')
                          , COALESCE(TRIM(UPPER(effective_specification_description)), '')
                          , COALESCE(TRIM(UPPER(measurement_unit)), '')
                      )
                    )
                  ELSE NULL
              END AS item_specification_key
        FROM estimate_rows
    )

    , ranked AS (
        SELECT
              *
            , ROW_NUMBER() OVER (
                PARTITION BY
                      project_id
                    , COALESCE(bid_item_sequence_number, -1)
                    , COALESCE(bid_code, '')
                    , COALESCE(CAST(specification_code AS STRING), '')
                    , COALESCE(bid_item_description, '')
                    , COALESCE(measurement_unit, '')
                ORDER BY
                      source_updated_at DESC NULLS LAST
                    , ingested_at_utc DESC NULLS LAST
                    , source_row_id DESC NULLS LAST
              ) AS estimate_item_rank

            , COUNT(*) OVER (
                PARTITION BY
                      project_id
                    , COALESCE(bid_item_sequence_number, -1)
                    , COALESCE(bid_code, '')
                    , COALESCE(CAST(specification_code AS STRING), '')
                    , COALESCE(bid_item_description, '')
                    , COALESCE(measurement_unit, '')
              ) AS estimate_item_version_count

            , MIN(engineer_estimate_unit_price) OVER (
                PARTITION BY
                      project_id
                    , COALESCE(bid_item_sequence_number, -1)
                    , COALESCE(bid_code, '')
                    , COALESCE(CAST(specification_code AS STRING), '')
                    , COALESCE(bid_item_description, '')
                    , COALESCE(measurement_unit, '')
              ) AS min_estimate_unit_price_for_item

            , MAX(engineer_estimate_unit_price) OVER (
                PARTITION BY
                      project_id
                    , COALESCE(bid_item_sequence_number, -1)
                    , COALESCE(bid_code, '')
                    , COALESCE(CAST(specification_code AS STRING), '')
                    , COALESCE(bid_item_description, '')
                    , COALESCE(measurement_unit, '')
              ) AS max_estimate_unit_price_for_item

            , MIN(engineer_project_total) OVER (
                PARTITION BY
                      project_id
                    , COALESCE(bid_item_sequence_number, -1)
                    , COALESCE(bid_code, '')
                    , COALESCE(CAST(specification_code AS STRING), '')
                    , COALESCE(bid_item_description, '')
                    , COALESCE(measurement_unit, '')
              ) AS min_engineer_project_total_for_item

            , MAX(engineer_project_total) OVER (
                PARTITION BY
                      project_id
                    , COALESCE(bid_item_sequence_number, -1)
                    , COALESCE(bid_code, '')
                    , COALESCE(CAST(specification_code AS STRING), '')
                    , COALESCE(bid_item_description, '')
                    , COALESCE(measurement_unit, '')
              ) AS max_engineer_project_total_for_item

        FROM estimate_rows_keyed
    )

    SELECT
          *
        , CASE WHEN estimate_item_version_count > 1 THEN TRUE ELSE FALSE END AS estimate_item_has_multiple_versions

        , CASE
              WHEN COALESCE(min_estimate_unit_price_for_item, CAST(-999999999.999 AS DECIMAL(38,10)))
                 <> COALESCE(max_estimate_unit_price_for_item, CAST(-999999999.999 AS DECIMAL(38,10)))
               OR COALESCE(min_engineer_project_total_for_item, CAST(-999999999.999 AS DECIMAL(38,10)))
                 <> COALESCE(max_engineer_project_total_for_item, CAST(-999999999.999 AS DECIMAL(38,10)))
              THEN TRUE ELSE FALSE
          END AS estimate_item_changed_across_versions

        , CONCAT_WS(
              '|'
            , COALESCE(project_id, '')
            , COALESCE(CAST(bid_item_sequence_number AS STRING), '')
            , COALESCE(bid_code, '')
            , COALESCE(CAST(specification_code AS STRING), '')
            , COALESCE(bid_item_description, '')
            , COALESCE(measurement_unit, '')
          ) AS estimate_item_key

    FROM ranked
    WHERE estimate_item_rank = 1
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, f"Built {TARGET_TABLE} successfully.")

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise